<a href="https://colab.research.google.com/github/Netrahoni/FlyRankAi-Intern-work-Files/blob/main/work/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Content Opportunity & Refresh Scoring Engine**
* **Internship Track:** Refresh / Content Opportunity Scoring
* **Author:** Netra Mani Pokhrel ([Netrahoni](https://github.com/Netrahoni))
* **Objective:** Build an honest machine learning pipeline using DuckDB and Scikit-Learn to identify decaying web pages and generate a prioritized editorial refresh playbook.

In [16]:
# ==========================================
# 1. ENVIRONMENT SETUP & AUTHENTICATION
# ==========================================
# Install DuckDB for high-performance querying over Parquet files
!pip install duckdb -q

import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Securely retrieve your Hugging Face read token from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    # Fallback input prompt if secrets manager is bypassed
    import getpass
    hf_token = getpass.getpass("Enter your Hugging Face Token: ")

# Initialize DuckDB and establish Hugging Face authentication
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}');")

print("Setup complete! DuckDB is authenticated and ready.")

Enter your Hugging Face Token: ··········
 Setup complete! DuckDB is authenticated and ready.


## **Step 2: Data Extraction & Feature Engineering**
We query the FlyRank warehouse release (`FlyRank/internship-warehouse`) using DuckDB.
* **Historical Feature Window:** November 1, 2025 – April 30, 2026 (6 months of baseline data).
* **Future Label Window:** May 1, 2026 – May 31, 2026 (1 month forward window).
* **Target Label (`needs_refresh`):** Binary flag set to `1` if future clicks dropped below 85% of the historical monthly average, signaling active content decay.

In [13]:
# ==========================================
# 2. SQL DATA EXTRACTION VIA DUCKDB
# ==========================================
query = """
WITH historical_features AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) as total_clicks,
        SUM(gsc_impressions) as total_impressions,
        AVG(gsc_avg_position) as avg_position,
        (SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions), 0)) as ctr
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2025-11-01' AND '2026-04-30'
    GROUP BY content_hash_id
),
future_labels AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) as future_clicks
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2026-05-01' AND '2026-05-31'
    GROUP BY content_hash_id
)
SELECT
    h.*,
    f.future_clicks,
    CASE WHEN f.future_clicks < (h.total_clicks / 6.0) * 0.85 THEN 1 ELSE 0 END as needs_refresh
FROM historical_features h
JOIN future_labels f ON h.content_hash_id = f.content_hash_id
WHERE h.total_impressions > 1000
"""

df = con.execute(query).df()
print(f"Success! Downloaded {df.shape[0]:,} rows and {df.shape[1]} columns of data.")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🚀 Success! Downloaded 79,223 rows and 7 columns of data.


,content_hash_id,total_clicks,total_impressions,avg_position,ctr,future_clicks,needs_refresh
0,content_d0fa1bbfbc10caf8,9.0,6035.0,17.074666,0.001491,3.0,0
1,content_4c1e972bec56132e,103.0,18509.0,10.923403,0.005565,137.0,0
2,content_64cad58fc02e7605,0.0,2869.0,39.545364,0.000000,0.0,0
3,content_4e48bd81bb37eb4f,37.0,57778.0,37.636799,0.000640,11.0,0
4,content_f338440914b1ab00,11.0,8099.0,18.312282,0.001358,5.0,0


## **Step 3: Model Training & Time-Aware Validation**
To prevent data leakage, we utilize a strict chronological split (80% training, 20% testing). We compare a simple **CTR Heuristic Baseline** against a robust **Random Forest Classifier** to ensure our model adds genuine predictive value over basic guessing.

In [14]:
# ==========================================
# 3. MACHINE LEARNING & EVALUATION
# ==========================================
# Clean missing values
df = df.fillna(0)

# Define features and labels
features = ['total_clicks', 'total_impressions', 'avg_position', 'ctr']
X = df[features]
y = df['needs_refresh']

# Time-aware split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline Evaluation (Heuristic: Low CTR = Decay)
average_ctr = X_train['ctr'].mean()
baseline_preds = (X_test['ctr'] < average_ctr).astype(int)

# Random Forest Model
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

print("--- BASELINE SCORECARD ---")
print(classification_report(y_test, baseline_preds, zero_division=0))

print("\n--- RANDOM FOREST MODEL SCORECARD ---")
print(classification_report(y_test, model_preds, zero_division=0))

--- BASELINE SCORECARD ---
              precision    recall  f1-score   support

           0       0.59      0.36      0.45      8704
           1       0.47      0.69      0.56      7141

    accuracy                           0.51     15845
   macro avg       0.53      0.53      0.51     15845
weighted avg       0.54      0.51      0.50     15845


--- RANDOM FOREST MODEL SCORECARD ---
              precision    recall  f1-score   support

           0       0.62      0.85      0.71      8704
           1       0.66      0.36      0.46      7141

    accuracy                           0.63     15845
   macro avg       0.64      0.60      0.59     15845
weighted avg       0.63      0.63      0.60     15845



##  **Step 4: The Ranked Action Playbook**
We convert model probabilities into business impact by calculating an **Opportunity Score**:
$$\text{Opportunity Score} = \text{Refresh Probability} \times \text{Total Impressions}$$
This guarantees that high-traffic pages at high risk of decay bubble straight to the top of the editorial team's queue.

In [15]:
# ==========================================
# 4. GENERATING THE ACTION PLAYBOOK
# ==========================================
probabilities = model.predict_proba(X_test)[:, 1]

results = X_test.copy()
results['content_hash_id'] = df.loc[X_test.index, 'content_hash_id']
results['refresh_probability'] = probabilities
results['opportunity_score'] = results['refresh_probability'] * results['total_impressions']

action_playbook = results.sort_values(by='opportunity_score', ascending=False)

print("TOP 10 CONTENT REFRESH TARGETS:")
action_playbook[['content_hash_id', 'total_impressions', 'refresh_probability', 'opportunity_score']].head(10)

🏆 TOP 10 CONTENT REFRESH TARGETS:


,content_hash_id,total_impressions,refresh_probability,opportunity_score
44525,content_acbcc847f8996314,638115.0,0.481922,307521.338561
19656,content_b556f0bd87d6fcca,619955.0,0.430280,266754.235137
46107,content_b17c1d1cb0a346d6,564466.0,0.421527,237937.787627
22915,content_519512944d7c341c,510900.0,0.459879,234951.947125
19745,content_c19eed2225ee5f40,518523.0,0.425122,220435.429087
7060,content_1e921148b5fee86a,435375.0,0.481922,209816.573464
38296,content_16b0867e41920435,508917.0,0.409628,208466.409181
19716,content_57dcb96896f9a33c,472460.0,0.410546,193966.351298
49276,content_5e1c049f62e33b11,390045.0,0.478771,186742.308078
39630,content_3b6e4c8d9a0a5c9c,386620.0,0.473775,183170.744232
